## PRÁCTICA 2: Aprendizaje Semi-supervisado y en Una Clase

### Cargar dataset

##### Trabajaremos con el conjunto de datos CIFAR-100 (50.000 instancias para entrenamiento y 10.000 para test), en el cual eliminaremos el 80% de sus etiquetas y realizaremos las siguientes particiones:

#####    40.000 instancias de entrenamiento no etiquetadas
#####    10.000 instancias de entrenamiento etiquetadas
#####    10.000 instancias de test etiquetadas

In [2]:
import tensorflow as tf
import numpy as np

In [3]:
labeled_data = 0.2 # Vamos a usar el etiquetado de sólo el 20% de los datos
np.random.seed(42)

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

indexes = np.arange(len(x_train))
np.random.shuffle(indexes)
ntrain_data = int(labeled_data*len(x_train))
unlabeled_train = x_train[indexes[ntrain_data:]]
x_train = x_train[indexes[:ntrain_data]]
y_train = y_train[indexes[:ntrain_data]]

In [7]:
print(x_train.shape)
print(unlabeled_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(10000, 32, 32, 3)
(40000, 32, 32, 3)
(10000, 1)
(10000, 32, 32, 3)
(10000, 1)


#### Preprocesado de los datos

In [ ]:
# convertimos de uint8 a float32 y normalizamos en rango [0, 1]
x_train = x_train.reshape(-1, 32, 32, 3).astype("float32") / 255.0
x_test = x_test.reshape(-1, 32, 32, 3).astype("float32") / 255.0


#### 1. Entrena un modelo, creado sobre TensorFlow, haciendo uso únicamente de las instancias etiquetadas de entrenamiento. Dicho modelo debe de tener al menos cuatro capas densas y/o convolucionales.

In [ ]:
miCNN = tf.keras.models.Sequential([
            tf.keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(32, 32, 3)),
            tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
            tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
            tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(100, activation="softmax")
        ])

optimizer = tf.keras.optimizers.Adam()

miCNN.compile(
            loss="sparse_categorical_crossentropy",
            optimizer=optimizer,
            metrics=["accuracy"]
        )

miCNN.fit(x_train, y_train, epochs=20, batch_size=32, verbose=1)


c:\Users\Usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.0352 - loss: 4.3750
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.1176 - loss: 3.7771
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.1864 - loss: 3.4102
Epoch 4/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.2307 - loss: 3.1622
Epoch 5/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.2783 - loss: 2.9328
Epoch 6/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.3147 - loss: 2.7199
Epoch 7/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.3534 - loss: 2.5298
Epoch 8/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.3946 - loss: 2.3426
Epoch 9/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.4376 - loss: 2.1568
Epoch 10/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4769 - loss: 1.9783
Epoch 11/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5193 - loss: 1.8056
Epoch 12/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step

#### Responde las siguientes preguntas:
####    a. ¿Qué red has escogido? ¿Por qué? ¿Cómo la has entrenado?
####    b. ¿Cuál es el rendimiento del modelo en entrenamiento? ¿Y en prueba?
####    c. ¿Qué conclusiones sacas de los resultados detallados en el punto anterior?

#### 2. Entrena el mismo modelo, incorporando las instancias no etiquetadas de entrenamiento mediante la técnica de auto-aprendizaje. Opcionalmente, se ponderará cada instancia de entrada en función de su calidad (o certeza).

In [ ]:
def miCNN_func():
    miCNN = tf.keras.models.Sequential([
                tf.keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(32, 32, 3)),
                tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
                tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
                tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
                tf.keras.layers.Flatten(),
                tf.keras.layers.Dense(128, activation="relu"),
                tf.keras.layers.Dense(100, activation="softmax")
            ])

    optimizer = tf.keras.optimizers.Adam()

    miCNN.compile(
                loss="sparse_categorical_crossentropy",
                optimizer=optimizer,
                metrics=["accuracy"]
            )
    
    return miCNN

In [61]:
def self_training(model_func, x_train, y_train, unlabeled_data, x_test, y_test, thresh=0.5, train_epochs=3):
    train_data = x_train.copy()
    train_label = y_train.copy()

    for i in range(train_epochs):
        model = model_func()
        print("Train Epoch: ", i)
        model.fit(train_data, train_label, epochs=5,
                  batch_size=128, verbose=1)

        y_pred = model.predict(unlabeled_data)

        y_class = np.argmax(y_pred, axis=1)
        y_value = np.max(y_pred, axis=1)

        train_data = x_train.copy()
        train_label = y_train.copy()

        for x_u, y_c, y_v in zip(unlabeled_data, y_class, y_value):
            if y_v > thresh:
                x_u = x_u[np.newaxis, ...]   # (32,32,3) → (1,32,32,3)
                train_data = np.concatenate((train_data, x_u), axis=0)
                train_label = np.append(train_label, y_c)

    return model

In [ ]:
model = self_training(
    miCNN_func,
    x_train,
    y_train,
    unlabeled_train,
    x_test,
    y_test,
    thresh=0.9,
    train_epochs=3
)

#### Responde las siguientes preguntas:
####    a. ¿Qué parámetros has definido para el entrenamiento?
####    b. ¿Cuál es el rendimiento del modelo en entrenamiento? ¿Y en prueba?
####    c. ¿Se mejoran los resultados obtenidos en el Ejercicio 1?
####    d. ¿Qué conclusiones sacas de los resultados detallados en los puntos anteriores?

#### 3. Entrena un modelo de aprendizaje semisupervisado de tipo autoencoder en dos pasos (primero el autoencoder, después el clasificador). La arquitectura del encoder debe ser exactamente la misma que la definida en los Ejercicios 1 y 2, a excepción del último bloque de capas.

In [ ]:
class MiAutoencoder:

    def __init__(self, input_shape):
        # encoder
        input_layer = tf.keras.layers.Input(shape=input_shape)
        x = tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same")(input_layer)
        x = tf.keras.layers.MaxPooling2D((2, 2), padding="same")(x)
        x = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
        code = tf.keras.layers.MaxPooling2D((2, 2), padding="same")(x)
        # decoder
        x = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same")(code)
        x = tf.keras.layers.UpSampling2D((2, 2))(x)
        x = tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
        x = tf.keras.layers.UpSampling2D((2, 2))(x)
        output_layer = tf.keras.layers.Conv2D(3, (3, 3), activation="sigmoid", padding="same")(x)

        self.autoencoder = tf.keras.Model(input_layer, output_layer)
        self.encoder = tf.keras.Model(input_layer, code)
        self.decoder = tf.keras.Model(code, output_layer)

        self.autoencoder.compile(optimizer="adam", loss="mse")

    def fit(self, X, y=None, sample_weight=None):
        self.autoencoder.fit(
            X, X,
            epochs=10,
            batch_size=128,
            shuffle=True,
            sample_weight=sample_weight,
            verbose=1
        )

    def get_encoded_data(self, X):
        return self.encoder.predict(X)

    def __del__(self):
        tf.keras.backend.clear_session()

In [ ]:
class MiClasificador:

    def __init__(self, input_dim):
        self.model = tf.keras.Sequential([
            tf.keras.layers.Dense(128, activation="relu", input_shape=(input_dim,)),
            tf.keras.layers.Dense(100, activation="softmax")
        ])

        self.model.compile(
            optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    def fit(self, X, y, sample_weight=None):
        self.model.fit(
            X, y,
            epochs=10,
            batch_size=128,
            shuffle=True,
            sample_weight=sample_weight,
            verbose=1
        )

    def predict(self, X):
        probs = self.model.predict(X)
        return np.argmax(probs, axis=1)

    def predict_proba(self, X):
        return self.model.predict(X)

    def score(self, X, y):
        loss, acc = self.model.evaluate(X, y, verbose=0)
        return acc

    def __del__(self):
        tf.keras.backend.clear_session()

In [46]:
def semisupervised_training(autoencoder, classifier, x_train, y_train, unlabeled_data):

    all_data = np.concatenate((x_train, unlabeled_data), axis=0)
    autoencoder.fit(all_data)
    code = autoencoder.get_encoded_data(x_train)
    classifier.fit(code, y_train)

    return autoencoder, classifier

In [ ]:
autoencoder = MiAutoencoder(input_shape=x_train.shape[1:])
classifier = MiClasificador(input_dim=64)  # tamaño del code

autoencoder, classifier = semisupervised_training(
    autoencoder,
    classifier,
    x_train,
    y_train,
    unlabeled_train
)

pred_data = autoencoder.get_encoded_data(x_test)
print("Test accuracy :", classifier.score(pred_data, y_test))

#### Responde las siguientes preguntas:
####    a. ¿Cuál es la arquitectura del modelo? ¿Y sus hiperparámetros?
####    b. ¿Cuál es el rendimiento del modelo en entrenamiento? ¿Y en prueba?
####    c. ¿Se mejoran los resultados obtenidos en los Ejercicios 1 y 2?
####    d. ¿Qué conclusiones sacas de los resultados detallados en los puntos anteriores?

#### 4. Entrena un modelo de aprendizaje semisupervisado de tipo autoencoder en un paso (autoencoder y clasificador al mismo tiempo). La arquitectura del autoencoder será la misma que la definida en el Ejercicio 3, y la combinación de encoder y clasificador será igual a la arquitectura definida en el Ejercicio 1.

In [ ]:
class MiClasificadorSemisupervisado:
    def __init__(self, input_shape):
        inputs = tf.keras.layers.Input(shape=input_shape)
        # encoder
        x = tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inputs)
        x = tf.keras.layers.MaxPooling2D((2, 2), padding="same")(x)
        x = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
        code = tf.keras.layers.MaxPooling2D((2, 2), padding="same")(x)
        # decoder
        x_decoder = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same")(code)
        x_decoder = tf.keras.layers.UpSampling2D((2, 2))(x_decoder)
        x_decoder = tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x_decoder)
        x_decoder = tf.keras.layers.UpSampling2D((2, 2))(x_decoder)
        decoding = tf.keras.layers.Conv2D(3, (3, 3), activation="sigmoid", padding="same", name="decoding")(x_decoder)
        # clasificador
        x_classifier = tf.keras.layers.Flatten()(code)
        x_classifier = tf.keras.layers.Dense(128, activation="relu")(x_classifier)
        classification = tf.keras.layers.Dense(100, activation="softmax", name="classification")(x_classifier)

        self.model = tf.keras.Model(inputs=inputs, outputs=[decoding, classification])

        self.model.compile(
            optimizer="adam",
            loss={"decoding": "mse", "classification": "sparse_categorical_crossentropy"},
            metrics={"classification": "accuracy"}
        )

    def fit(self, X, y, unlabeled_data):
        X_all = np.concatenate((X, unlabeled_data), axis=0)

        y_decoding = X_all    # la etiqueta es la propia entrada
        y_classification = np.concatenate((y.reshape(-1), np.zeros(len(unlabeled_data))))

        w_classification = np.concatenate((np.ones(len(X)), np.zeros(len(unlabeled_data))))

        self.model.fit(
            X_all,
            [y_decoding, y_classification],
            sample_weight=[None, w_classification],
            epochs=20,
            batch_size=256,
            shuffle=True,
            verbose=1
        )

    def predict(self, X):
        _, probs = self.model.predict(X)
        return np.argmax(probs, axis=1)

    def predict_proba(self, X):
        _, probs = self.model.predict(X)
        return probs

    def score(self, X, y):
        preds = self.predict(X)
        return np.mean(preds == y)

    def __del__(self):
        tf.keras.backend.clear_session()

In [75]:
def semisupervised_training_v2(model, x_train, y_train, unlabeled_data):
    model.fit(x_train, y_train, unlabeled_data)
    return model

In [ ]:
model = MiClasificadorSemisupervisado(input_shape=x_train.shape[1:])

model = semisupervised_training_v2(
    model,
    x_train,
    y_train,
    unlabeled_train
)

print(model.score(x_test, y_test))

#### Responde las siguientes preguntas:
####    a. ¿Cuál es la arquitectura del modelo? ¿Y sus hiperparámetros?
####    b. ¿Cuál es el rendimiento del modelo en entrenamiento? ¿Y en prueba?
####    c. ¿Se mejoran los resultados obtenidos en los ejercicios anteriores?
####    d. ¿Qué conclusiones sacas de los resultados detallados en los puntos anteriores?

#### 5. Repite el mismo entrenamiento de los Ejercicios 2-4, pero eliminando las instancias no etiquetadas más atípicas con respecto a los datos etiquetados. Se cumplirán los siguientes puntos:

#### · La arquitectura de la red de clasificación en una clase será la misma a la utilizada en el clasificador del Ejercicio 1, a excepción de la capa de salida.

#### ·  Utiliza la técnica explicada en el Notebook 5, usando un valor de 𝑣 = 0,9.

#### Responde las siguientes preguntas:
####    a. ¿Se mejoran los resultados con respecto a los anteriores ejercicios? ¿Qué conclusiones sacas de estos resultados?

#### 6. Repite los Ejercicios 3-5 cambiando el autencoder por la técnica definida en el apartado “Hay vida más allá del autoencoder” del Notebook 4. Contesta a las preguntas de dichos ejercicios. Se cumplirán los siguientes puntos:

#### Responde las siguientes preguntas:
####    a. La arquitectura de la red será igual a la parte encoder del autencoder definido en los ejercicios anteriores
####    b. El modelo debe entrenar correctamente.